<a href="https://colab.research.google.com/github/Devansh63/Language-Model-From-Scratch/blob/main/Copy_of_MP2_Language_Model_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spring 2026: CS446/ECE449 MP2: Build Your Language Model From Scratch

MP Development TA: Yuxi Chen  
Instructor: Prof. Huan Zhang

Note: Please use **"Copy to Drive”** to create your own copy of the notebook so that your changes are saved.

## ✨ Introduction

In this MP, you are building a minimal language-model pipeline from the ground up. By the end, your code will take raw text, turn it into tokens, train a decoder-only Transformer, and generate new text one token at a time.

### Notebook Workflow

1. Run the setup cell.
2. Run the download cell to fetch the full MP files from the course site.
3. Edit the `%%writefile` cells to fill in the TODO sections for each task. A `%%writefile` cell saves everything in the cell into a file (like a .py file) instead of running it.
4. After each task, rerun the edited `%%writefile` cell and then run the corresponding visible test.
5. After each component passes its individual tests, run the full visible tests to verify everything works together.
6. Then train, try inference, and export your submission zip.

### Environment Setup and Library Policy

This MP uses PyTorch for tensor computation and training utilities, but the model logic itself must be implemented by you. If you are using Google Colab, please select the runtime backend `2026.01` to ensure proper package versions. Please only use the libraries specified in the allowed list below. The goal of this MP is to implement the core components of a language model yourself, so using high-level libraries that abstract away these details is not permitted.

**Allowed:** PyTorch tensor operations, PyTorch modules already present in the starter code, `torch.optim.Adam` as the optimizer, and the Python standard library.

**Disallowed:** Do not use high-level external model/tokenizer libraries including but not limited to:

- `transformers`
- `tokenizers`
- `sentencepiece`
- `torch.nn.Transformer*`
- `torch.nn.CrossEntropyLoss`
- other libraries that implement the core logic of tokenization, attention, or training for you.

<div class="alert alert-block alert-info">
<b>Note:</b> This MP involves computationally intensive training and may run slowly on standard free Colab sessions due to limited resources.
We <b>strongly recommend</b> using the <b>Colab Pro for Education</b>, which provides access to more powerful GPUs and longer runtimes, making the MP significantly smoother to complete. You can sign up for free here (Colab Pro is <b>free</b> for students):
<a href="https://colab.research.google.com/signup" target="_blank">https://colab.research.google.com/signup</a>
</div>


## 🚀 Get Started on Google Colab

### ☁️ How Colab Works

Colab runs your code on **cloud machines**, not your own computer.  
The files you see (left sidebar → **Files**) are also stored in the cloud and may be temporary.

📺 Quick intro (optional):  
https://www.youtube.com/watch?v=RLYoEyIHL6A

---

### ⚙️ Select GPU Runtime in Colab

1. Click **Runtime** (top menu)  
2. Click **Change runtime type**  
3. Set **Hardware accelerator → GPU**  
4. Click **Save**

If available, we recommend using a **L4 GPU instance**. This is included with the free **Colab for Education** subscription and provides good performance for this MP. If you do not have access to an L4 GPU, a T4 GPU will also work, but training may be slower.

<img src="https://courses.grainger.illinois.edu/cs446/sp2026/mp2_files/images/runtime.png" width="50%">

## 📚 Setup Workspace

Run the next cell once per fresh Colab runtime. It creates the workspace folders, and configures the runtime environment for the MP. You can rerun it later if you accidentally delete any of the folders or files, but you only need to run it once at the start.

The notebook expects to work with this layout:

- `mp/`
- `tests_visible/`
- `data/`
- `mp/artifacts/`


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

for rel in ["mp", "mp/artifacts", "data", "tests_visible"]:
    Path(rel).mkdir(parents=True, exist_ok=True)

if importlib.util.find_spec("pytest") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytest"])

workspace_root = str(Path('.').resolve())
existing_pythonpath = os.environ.get("PYTHONPATH", "")
pythonpath_parts = [part for part in existing_pythonpath.split(":") if part]
if workspace_root not in pythonpath_parts:
    pythonpath_parts.insert(0, workspace_root)
os.environ["PYTHONPATH"] = ":".join(pythonpath_parts)

print("Workspace ready.")

## 💾 Download MP Files

Run the next cell to download the full MP files from the course website. This includes the starter code, test files, and datasets. You can rerun it later if you accidentally delete any of the folders or files, but you only need to run it once at the start.

After downloading, you can view the files by clicking the **📁 Files icon** on the left sidebar in Colab. You should see folders like `mp/`, `data/`, and `tests_visible` there.

Note that Colab runs in the **cloud**, not on your local computer:
- These files are stored in the Colab runtime, not on your own device.
- They are temporary and tied to your current session.
- If your runtime disconnects or resets, the files may be deleted—just rerun the download cell to restore them.

In [ ]:
from pathlib import Path
import urllib.request

COURSE_FILES_BASE_URL = "https://courses.grainger.illinois.edu/cs446/sp2026/mp2_files"

STARTER_FILES = [
    "mp/__init__.py",
    "mp/tokenizer.py",
    "mp/model.py",
    "mp/train.py",
    "mp/run_train.py",
    "mp/serve.py",
    "mp/check_perplexity.py"
]

VISIBLE_TEST_FILES = [
    "tests_visible/conftest.py",
    "tests_visible/test_api_scaffold.py",
    "tests_visible/test_tokenizer.py",
    "tests_visible/test_model.py",
    "tests_visible/test_train.py",
]

DATA_FILES = [
    "data/tiny_corpus.txt",
    "data/tinystories_subset.txt",
]


def _download_one(rel_path: str, overwrite: bool) -> None:
    destination = Path(rel_path)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and not overwrite:
        print(f"Keeping existing {rel_path}")
        return

    url = f"{COURSE_FILES_BASE_URL.rstrip('/')}/{rel_path.lstrip('/')}"
    try:
        with urllib.request.urlopen(url) as response:
            destination.write_bytes(response.read())
        print(f"Downloaded {rel_path} <- {url}")
    except Exception as exc:
        raise RuntimeError(f"Could not download {rel_path} from {url}: {exc}") from exc


def download_mp_files(*, overwrite: bool = False, include_tests: bool = True, include_data: bool = True) -> None:
    files = list(STARTER_FILES)
    if include_tests:
        files.extend(VISIBLE_TEST_FILES)
    if include_data:
        files.extend(DATA_FILES)
    for rel_path in files:
        _download_one(rel_path, overwrite=overwrite)

download_mp_files(overwrite=False, include_tests=True, include_data=True)

## ⚒️ Task #1: Byte-Level BPE Tokenization

Before a model can train on text, the text must become token IDs. BPE (Byte Pair Encoding) is a tokenization algorithm that starts from very small units and gradually learns larger recurring chunks by merging frequent adjacent pairs.

In this MP, the basic units are bytes:

- token IDs `0..255` correspond to raw byte values
- learned merges create new token IDs starting at `256`

This byte-level setup is useful because:

- it works for arbitrary UTF-8 text
- it avoids special handling for unknown words
- it keeps the tokenizer conceptually simple

You can think of BPE training as teaching the tokenizer which byte patterns travel together often enough to deserve their own shared symbol.

### What BPE Training Does

Suppose the text is:

```text
abababa
```

UTF-8 bytes:

```text
[97, 98, 97, 98, 97, 98, 97]
```

Now count adjacent pairs:

```text
(97, 98) -> 3
(98, 97) -> 3
```

Both pairs appear 3 times. To keep training deterministic, the tie is broken by choosing the lexicographically smaller pair. So `(97, 98)` is merged first.

Create a new token ID `256` and replace every non-overlapping occurrence:

```text
[97, 98, 97, 98, 97, 98, 97]
-> [256, 256, 256, 97]
```

Then repeat:

1. recompute pair frequencies on the updated sequence
2. choose the best pair
3. create the next token ID
4. replace all non-overlapping occurrences

### Why Merge Order Matters

The merge list defines the priority used later in `encode`. If one pair was learned earlier than another, it should take precedence during encoding.

### Why Non-Overlapping Replacement Matters

Suppose the current sequence is:

```text
[1, 1, 1]
```

and the pair is `(1, 1)`.

The correct replacement is:

```text
[256, 1]
```

not:

```text
[256, 256]
```

because the second merge would overlap the first one.

### Encoding and Decoding

After training:

- `encode` should start from UTF-8 bytes and repeatedly apply the highest-priority learned merge that is currently present
- `decode` should recover the underlying bytes for each token ID, concatenate them, and decode them back into text

### What You Need to Implement

- `BPETokenizer.train`
- `BPETokenizer.encode`
- `BPETokenizer.decode`

Edit the next cell and rerun it to write your updated tokenizer file.


In [ ]:
%%writefile mp/tokenizer.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Callable


class BPETokenizer:
    """A minimal byte-level BPE tokenizer.

    Base vocabulary ids 0-255 correspond to raw byte values.
    Learned merges add new token ids starting at 256.
    """

    def __init__(self, merges: list[tuple[int, int]]) -> None:
        self.merges = merges
        self.base_vocab_size = 256
        self.vocab_size = self.base_vocab_size + len(merges)

        # Map pair -> rank (merge order).
        self.merge_ranks: dict[tuple[int, int], int] = {
            pair: rank for rank, pair in enumerate(merges)
        }

        # Map token id -> raw bytes represented by that token.
        self.id_to_bytes: dict[int, bytes] = {
            i: bytes([i]) for i in range(self.base_vocab_size)
        }
        for idx, (left, right) in enumerate(merges, start=self.base_vocab_size):
            self.id_to_bytes[idx] = self.id_to_bytes[left] + self.id_to_bytes[right]

    @classmethod
    def train(
        cls,
        text: str,
        vocab_size: int,
        min_pair_freq: int = 2,
        progress_callback: Callable[[int, int, tuple[int, int], int], None] | None = None,
    ) -> "BPETokenizer":
        """Train a byte-level BPE tokenizer from text.

        TODO:
        Implement standard byte-level BPE training with deterministic behavior.

        Required behavior:
        1) Start from the UTF-8 bytes of the training text, where each byte is one token.
        2) Repeatedly count frequencies of adjacent token pairs in the current token sequence.
        3) Pick exactly one pair to merge each round:
           - prefer higher frequency
           - break ties by lexicographically smaller pair
        4) Stop if:
           - the vocabulary would exceed `vocab_size`, or
           - no pair appears at least `min_pair_freq` times, or
           - the sequence is too short to contain a pair
        5) When merging, replace every non-overlapping occurrence of the chosen pair
           with the next token id (starting at 256, then 257, ...).
        6) Record merges in the order they are learned and return `BPETokenizer(merges)`.

        Edge cases to handle:
        - empty input text should produce a valid tokenizer with no merges
        - `vocab_size < 256` should not silently produce invalid token ids
        - repeated runs on the same text must learn the same merge sequence

        Optional progress reporting:
        - if `progress_callback` is provided, call it after each learned merge with:
          `progress_callback(num_merges_learned, total_possible_merges, best_pair, best_count)`
        """
        raise NotImplementedError

    def encode(self, text: str) -> list[int]:
        """Encode text to token ids using learned merges.

        TODO:
        Apply the learned merges greedily by merge priority.

        Required behavior:
        1) Convert the input text to its UTF-8 byte sequence.
        2) Repeatedly scan the current token sequence for adjacent pairs that were learned
           during training.
        3) If multiple learned pairs are present, choose the one with the smallest merge rank
           (the pair learned earliest).
        4) Merge one occurrence of that best-ranked pair, update the sequence, and continue.
        5) Stop only when no adjacent pair in the sequence is mergeable.

        Important details:
        - start from raw bytes every time; do not reuse training-time state
        - preserve the exact byte content of the string
        - empty input should return an empty list
        """
        raise NotImplementedError

    def decode(self, ids: list[int]) -> str:
        """Decode token ids back to text.

        TODO:
        Reconstruct the original byte stream and decode it as UTF-8.

        Required behavior:
        1) Look up the byte representation for each token id using `self.id_to_bytes`.
        2) Concatenate those byte chunks in order.
        3) Decode the result as UTF-8.

        Important details:
        - decoding should succeed even if byte boundaries are imperfect
        - empty input should return the empty string
        - the output should round-trip correctly for normal UTF-8 text
        """
        raise NotImplementedError

    def save(self, path: str) -> None:
        """Save tokenizer merges to JSON."""
        output_path = Path(path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", encoding="utf-8") as f:
            json.dump({"merges": self.merges}, f)

    @classmethod
    def load(cls, path: str) -> "BPETokenizer":
        """Load tokenizer merges from JSON and reconstruct a BPETokenizer."""
        with Path(path).open("r", encoding="utf-8") as f:
            payload = json.load(f)
        merges_raw = payload.get("merges", [])
        merges = [tuple(pair) for pair in merges_raw]
        return cls(merges)

    @staticmethod
    def _replace_pair(
        token_ids: list[int],
        pair: tuple[int, int],
        new_token_id: int,
    ) -> list[int]:
        """Replace all non-overlapping occurrences of `pair` with `new_token_id`."""
        result: list[int] = []
        i = 0
        while i < len(token_ids):
            if (
                i + 1 < len(token_ids)
                and token_ids[i] == pair[0]
                and token_ids[i + 1] == pair[1]
            ):
                result.append(new_token_id)
                i += 2
            else:
                result.append(token_ids[i])
                i += 1
        return result


### Task #1 Tests

Once you have implemented `BPETokenizer.train`, `BPETokenizer.encode`, and `BPETokenizer.decode`, run the tokenizer test below. If you run into issues, check the **Debugging Notes** section at the end of the notebook for common pitfalls and guidance. You can check the test file `tests_visible/test_tokenizer.py` for the expected behavior of the tokenizer on various inputs. If your implementation is correct, all assertions in the test should pass without errors.


In [ ]:
!pytest tests_visible/test_tokenizer.py -q --capture=tee-sys

## ⚒️ Task #2: Decoder-Only Transformer

Once text becomes token IDs, the model must predict what token comes next.

This MP uses a decoder-only Transformer. Decoder-only means:

- each position sees itself and earlier positions
- no position is allowed to attend to future tokens
- the model predicts next-token distributions over a vocabulary

You can think of the Transformer as a stack of context-building layers. Each layer lets every position gather information from earlier positions, refine its representation, and hand a richer version forward.

### What the Model Does

At a high level:

1. token IDs are mapped to token embeddings
2. positions are mapped to fixed sinusoidal encodings
3. those embeddings are added together
4. the result passes through a stack of Transformer blocks
5. the final hidden states are projected to vocabulary logits

So the model turns:

```text
[batch, seq_len]
```

into:

```text
[batch, seq_len, vocab_size]
```

### Sinusoidal Positional Encoding

This MP uses fixed sinusoidal positional encoding instead of learned position
embedding parameters.

For position `pos` and hidden dimension `i`, the encoding is:

```text
PE[pos, 2i]   = sin(pos / 10000^(2i / d_model))
PE[pos, 2i+1] = cos(pos / 10000^(2i / d_model))
```

This gives every position a deterministic pattern of values across the hidden
dimension. Nearby positions have similar encodings, and the model can use
different frequencies to reason about both short-range and longer-range order.

In `mp/model.py`, you will implement this in a helper function and then add the
result to the token embeddings inside `TransformerLM.forward`.

### Causal Self-Attention

The most important idea in the model is self-attention.

For every position, the model produces:

- a query: what kind of information am I looking for?
- a key: what information do I contain?
- a value: what information should be passed along if I am selected?

Queries interact with keys to produce attention scores. Those scores are normalized into weights, and the weighted values are combined into a contextual representation for each position.

### Why the Causal Mask Is Essential

During training, the full sequence is available in memory. But a language model is not allowed to use future tokens when predicting the next token.

For example, if the model is processing:

```text
The rabbit opened the
```

it should not be able to peek at the next word and cheat.

That is why attention scores must be masked before normalization so that position `t` cannot attend to positions `> t`.

### Multi-Head Attention

Instead of using a single attention pattern, the model splits the hidden dimension into multiple heads.

This lets different heads focus on different kinds of information, such as:

- local phrase structure
- repeated names or entities
- longer-range dependencies

Each head computes attention separately, and the heads are merged back together afterward.

### Transformer Blocks

A Transformer block contains two residual sublayers:

1. self-attention
2. feed-forward network

This MP uses pre-layernorm structure:

```text
x = x + attention(layernorm(x))
x = x + feed_forward(layernorm(x))
```

Residual connections help preserve information and make training more stable.

### Generation

Once the model is trained, generation works autoregressively:

1. run the model on the current context
2. read the logits at the final position
3. choose the next token
4. append it to the sequence
5. repeat

This is implemented in `TransformerLM.generate`.

### Temperature and Top-k

Two common decoding controls appear in this MP:

#### Temperature

Temperature changes how sharp the next-token distribution is.

- low temperature: more confident, less random
- high temperature: flatter, more random
- `temperature <= 0` means use greedy decoding in this MP

If the model strongly prefers one token, lowering temperature makes that preference even stronger. Raising temperature gives lower-scoring tokens more chance to be sampled.

#### Top-k

Top-k sampling keeps only the `k` highest-scoring candidate tokens and discards the rest before sampling.

For example:

- `top_k = 5` means sample only from the best 5 candidates
- `top_k = 1` is effectively greedy decoding

This is useful because it prevents extremely unlikely tokens from being sampled.

### What You Need to Implement

- `sinusoidal_position_encoding`
- `CausalSelfAttention.forward`
- `TransformerBlock.forward`
- `TransformerLM.forward`
- `TransformerLM.generate`

Edit the next cell and rerun it to write your updated model file.


In [ ]:
%%writefile mp/model.py
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
from torch import nn


@dataclass
class TransformerConfig:
    vocab_size: int
    max_seq_len: int
    n_layers: int
    n_heads: int
    d_model: int
    d_ff: int
    dropout: float


def sinusoidal_position_encoding(
    seq_len: int,
    d_model: int,
) -> torch.Tensor:
    """Return sinusoidal positional encodings with shape [1, T, d_model].

    TODO:
    Implement the fixed sinusoidal positional encoding used in the original
    Transformer paper.

    Required behavior:
    1) Create position indices `0, 1, ..., seq_len - 1`.
    2) For every even hidden dimension, compute:
       `sin(position / 10000^(i / d_model))`
    3) For every odd hidden dimension, compute:
       `cos(position / 10000^(i / d_model))`
    4) Return the encodings with shape [1, seq_len, d_model].

    Important details:
    - return dtype should be `torch.float32`
    - you may assume `d_model` is even in this MP
    """
    raise NotImplementedError


class CausalSelfAttention(nn.Module):
    def __init__(self, config: TransformerConfig) -> None:
        super().__init__()
        if config.d_model % config.n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")

        self.n_heads = config.n_heads
        self.d_model = config.d_model
        self.head_dim = config.d_model // config.n_heads

        self.qkv_proj = nn.Linear(config.d_model, 3 * config.d_model)
        self.out_proj = nn.Linear(config.d_model, config.d_model)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute masked multi-head self-attention.

        TODO:
        Implement multi-head causal self-attention for inputs of shape [B, T, d_model].

        Required behavior:
        1) Produce query, key, and value vectors for every token position.
        2) Split the model dimension into `n_heads` heads of size `head_dim`.
        3) Compute attention scores between every query position and every key position.
        4) Scale scores so they do not grow too large as `head_dim` increases.
        5) Apply a causal mask before normalization so position `t` can only attend to
           positions `<= t`.
        6) Normalize scores across the last dimension, apply attention dropout, and use the
           resulting weights to combine value vectors.
        7) Merge heads back to shape [B, T, d_model], apply the output projection, then apply
           residual dropout.

        Important details:
        - output shape must match the input shape
        - the mask must work for any sequence length up to `max_seq_len`
        - changing future tokens must not change logits for earlier positions
        """
        raise NotImplementedError


class TransformerBlock(nn.Module):
    def __init__(self, config: TransformerConfig) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.ff = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff),
            nn.GELU(),
            nn.Linear(config.d_ff, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply pre-LN attention + MLP with residual connections.

        TODO:
        Implement the standard pre-layernorm Transformer block.

        Required behavior:
        1) Normalize the input before the attention sublayer.
        2) Add the attention output back to the original residual stream.
        3) Normalize the updated hidden states before the feed-forward sublayer.
        4) Add the feed-forward output back to the residual stream.

        Important details:
        - preserve shape [B, T, d_model]
        - both sublayers should be residual updates, not replacements
        """
        raise NotImplementedError


class TransformerLM(nn.Module):
    def __init__(self, config: TransformerConfig) -> None:
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.drop = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Return logits with shape [B, T, vocab_size].

        TODO:
        Build the full decoder-only language model forward pass.

        Required behavior:
        1) Treat `input_ids` as shape [batch, seq_len].
        2) Reject sequences longer than `config.max_seq_len`.
        3) Build token embeddings and sinusoidal positional encodings for each time step.
        4) Add token and position information, then apply embedding dropout.
        5) Run the hidden states through every Transformer block in order.
        6) Apply the final layer norm and project to vocabulary logits.

        Important details:
        - output must have shape [B, T, vocab_size]
        - logits at each position correspond to the next-token prediction distribution
        """
        raise NotImplementedError

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
    ) -> torch.Tensor:
        """Autoregressively generate new tokens.

        TODO:
        Extend the prompt one token at a time.

        Required behavior:
        1) Repeatedly run the model on the current context and read the logits from the final
           time step only.
        2) If the current sequence is longer than `max_seq_len`, use only the most recent
           `max_seq_len` tokens as context.
        3) Choose the next token by:
           - greedy decoding when `temperature <= 0` or `top_k == 1`
           - otherwise sampling from the temperature-scaled distribution
        4) If `top_k` is provided, only allow sampling from the top-k logits.
        5) Append the chosen token and continue until `max_new_tokens` tokens are added.

        Important details:
        - return the full sequence including the original prompt
        - generation should work for batch size 1 and larger
        - greedy decoding and `top_k == 1` should behave identically
        """
        raise NotImplementedError


### Task #2 Tests

Once you have implemented the Transformer forward pass and generation logic, run the model test below. If you run into issues, check the **Debugging Notes** section at the end of the notebook for common pitfalls and guidance. You can check the test file `tests_visible/test_model.py` for the expected behavior of the model on various inputs. If your implementation is correct, all assertions in the test should pass without errors.


In [ ]:
!pytest tests_visible/test_model.py -q --capture=tee-sys

## ⚒️ Task #3: Cross-Entropy and Next-Token Training

The model outputs logits, not probabilities. To train it, you need a loss function that penalizes the model when it assigns low score to the correct next token.

In the course slides, we define the training loss as the **negative log-likelihood (NLL)**: $-\log P_\theta(x_t \mid x_{<t})$. In this assignment, we compute the loss from **logits** using $\log \sum_j \exp(z_{t,j}) - z_{t,\text{correct}}$, where $z_{t,j}$ is the **logit** (unnormalized score) for token $j$ at position $t$.

These are **exactly equivalent**: this is just the NLL written in terms of logits instead of probabilities (via the softmax).

For one position:

```text
loss = log(sum(exp(all logits))) - logit_of_correct_class
```

Then the loss is averaged across all positions in the batch.

### Why the Target Is Shifted

If the sequence is:

```text
[A, B, C, D]
```

then the model input and target are:

```text
x = [A, B, C]
y = [B, C, D]
```

So `get_batch` should sample windows from a 1D token stream and create:

- `x`: current-token windows
- `y`: next-token windows shifted by one position

### What a Training Step Does

1. put the model in training mode
2. clear old gradients
3. run a forward pass
4. compute cross-entropy
5. run backpropagation
6. update parameters with the optimizer

### What You Need to Implement

- `cross_entropy_from_logits`
- `get_batch`
- `train_step`

### Important Cases to Think About

- logits with arbitrary leading dimensions
- gathering the correct class at every position
- making sure `x` and `y` are shifted exactly by one token
- returning a Python float from `train_step`

Edit the next cell and rerun it to write your updated training file.


In [ ]:
%%writefile mp/train.py
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
from typing import Any

import torch

from .model import TransformerConfig


def cross_entropy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Cross-entropy from logits using logsumexp + gather.

    TODO:
    Compute next-token cross-entropy directly from logits.

    Required behavior:
    1) Treat the last dimension of `logits` as the vocabulary dimension.
    2) For every position, compute:
       log(normalizer over all classes) - logit of the target class
    3) Average the per-position losses into one scalar tensor.

    Important details:
    - this must work for logits with any number of leading batch dimensions
    - `targets` has the same leading shape as `logits[..., 0]`
    - the result should numerically match the standard cross-entropy definition
    """
    raise NotImplementedError


def get_batch(
    token_ids: torch.Tensor,
    batch_size: int,
    seq_len: int,
    device: str,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Create random next-token prediction batches.

    TODO:
    Sample training examples from a 1D token stream.

    Required behavior:
    1) Randomly choose `batch_size` starting positions in `token_ids`.
    2) For each start position, create:
       - an input sequence of length `seq_len`
       - a target sequence of length `seq_len` shifted by one token to the right
    3) Stack all sampled examples into tensors of shape [B, T].
    4) Move the returned tensors to `device`.

    Important details:
    - `token_ids` is a single long 1D sequence, not a batch
    - every target token should be the next token after the corresponding input token
    - sampled windows must stay within bounds
    """
    raise NotImplementedError


def train_step(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    x: torch.Tensor,
    y: torch.Tensor,
) -> float:
    """Run one optimization step and return scalar loss.

    TODO:
    Execute one full training iteration.

    Required behavior:
    1) Put the model in training mode.
    2) Clear old gradients before the new forward/backward pass.
    3) Run the model on `x` and compute the cross-entropy loss against `y`.
    4) Backpropagate through the loss.
    5) Update parameters with the optimizer.
    6) Return the loss as a Python float.

    Important details:
    - do not skip the backward pass or optimizer step
    - the returned value should be easy to log or append to a list
    - the function should work on CPU and GPU
    """
    raise NotImplementedError


@torch.no_grad()
def evaluate(
    model: torch.nn.Module,
    val_tokens: torch.Tensor,
    batch_size: int,
    seq_len: int,
    eval_iters: int,
) -> float:
    """Evaluate average validation loss over eval_iters mini-batches."""
    model.eval()
    device = str(next(model.parameters()).device)

    losses = []
    for _ in range(eval_iters):
        x, y = get_batch(val_tokens, batch_size=batch_size, seq_len=seq_len, device=device)
        logits = model(x)
        losses.append(float(cross_entropy_from_logits(logits, y).item()))
    return sum(losses) / len(losses)


def save_checkpoint(
    path: str,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    step: int,
    config: TransformerConfig,
) -> None:
    """Save training state."""
    payload: dict[str, Any] = {
        "step": step,
        "config": asdict(config),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
    }
    out_path = Path(path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, out_path)


def load_checkpoint(
    path: str,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, Any]:
    """Load training state into model (+ optimizer if provided)."""
    payload = torch.load(path, map_location="cpu")
    model.load_state_dict(payload["model_state"])
    if optimizer is not None and "optimizer_state" in payload:
        optimizer.load_state_dict(payload["optimizer_state"])
    return {"step": payload.get("step", 0), "config": payload.get("config", {})}


### Task #3 Tests

Once you have implemented cross-entropy, batching, and a training step, run the training-utilities test below. If you run into issues, check the **Debugging Notes** section at the end of the notebook for common pitfalls and guidance. You can check the test file `tests_visible/test_train.py` for the expected behavior of the training utilities on various inputs. If your implementation is correct, all assertions in the test should pass without errors.    


In [ ]:
!pytest tests_visible/test_train.py -q --capture=tee-sys

## 🧪 Full Visible Tests

Congratulations on completing all the MP implementation tasks! 🎉  

Use the next cell to run the full set of visible test cases. This step verifies that all components work together correctly and helps ensure your model can train and generate outputs as expected before you proceed to final training and submission.

Keep in mind that the hidden tests will further evaluate:
- Whether your transformer correctly functions as a **causal language model**
- Whether your tokenizer is **deterministic** and can reliably encode/decode text
- Whether your training utilities implement proper **next-token prediction behavior**

In [ ]:
!pytest tests_visible/test_api_scaffold.py tests_visible/test_tokenizer.py tests_visible/test_model.py tests_visible/test_train.py -q --capture=tee-sys

## 🚂 Training Your Model

Once the visible tests are passing, run the following cells to train in the notebook runtime. The training should take around 60 minutes on a T4 instance, depending on the runtime environment. The training script saves the model checkpoint and tokenizer artifacts to `mp/artifacts/`, which are required for the hidden training-artifact test.

If available, we recommend using a **L4 GPU instance**. This is included with the free **Colab for Education** subscription and provides good performance for this MP. If you do not have access to an L4 GPU, a T4 GPU will also work, but training may be slower.

Training uses:

- `data/tinystories_subset.txt`
- provided training script `mp/run_train.py` (The script uses a provided efficient tokenizer implementation so end-to-end training stays practical on larger corpora. This means the actual training run does **not** depend on your tokenizer TODO implementation. However, you still need to implement the tokenizer TODOs, and those functions are graded by the tests.)

<div class="alert alert-block alert-info">
<b>Hidden Training-Artifact Test:</b>  
The hidden grader evaluates the quality of your trained model. Your submission must include the following files:

<ul>
  <li><code>mp/artifacts/checkpoint.pt</code></li>
  <li><code>mp/artifacts/tokenizer.json</code></li>
</ul>

These files are generated by running the training cells below. The grader will load your tokenizer and checkpoint and <b>compute perplexity</b> on a hidden TinyStories validation subset. Your model must achieve a perplexity that is meaningfully better than a random baseline to receive full credit.
</div>

In [ ]:
from pathlib import Path
subset_path = Path("data/tinystories_subset.txt")
if not subset_path.exists():
    raise FileNotFoundError(
        f"Missing {subset_path}. Rerun the download cell above to fetch the course-provided subset."
    )

print(f"Using {subset_path}")
print("If you only want a quick workflow smoke test, you may temporarily change data/tinystories_subset.txt to data/tiny_corpus.txt, and then restore the course-provided subset before creating final artifacts.")

In [ ]:
!python -m mp.run_train

## 🩺 Local Perplexity Sanity Check

After training, run the next cell to calculate perplexity on a public sanity-check slice from `data/tinystories_subset.txt`.

As a rough guideline, try to reach a perplexity below **120** on this local check before creating your submission. This helps confirm that your model is learning meaningful patterns. Keep in mind that the grader will evaluate your checkpoint and tokenizer on a different TinyStories subset, so passing this check is not a strict guarantee of final performance.

In [ ]:
!python -m mp.check_perplexity

## ⚡ Try Inference With Your Trained Model

After training completes, run the next cell to generate text from your saved checkpoint and tokenizer.

Try a few prompts of your own, for example:

- `"The little rabbit said"`
- `"In the dark forest"`
- `"Mia opened the door and"`

If training and generation are working, the output may still be imperfect since we are training on a small dataset, but it should look like plausible story text rather than random characters or broken token fragments.

If you're interested in further improving quality, you can try training your model on the full [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) dataset, which provides significantly more data.


In [ ]:
!python -m mp.serve \
  --checkpoint mp/artifacts/checkpoint.pt \
  --tokenizer mp/artifacts/tokenizer.json \
  --prompt "Once upon a time" \
  --max-new-tokens 40 \
  --temperature 0.7 \
  --top-k 20

## 📝 Generating Your Submission Zip

Submit a `.zip` whose top-level contains `mp/`.

Required structure:

```text
mp/
  __init__.py
  tokenizer.py
  model.py
  train.py
  run_train.py
  serve.py
  artifacts/
    checkpoint.pt
    tokenizer.json
```

Important:

- do not submit `mp` nested inside another folder
- do not omit the files in `mp/artifacts/`
- the grader imports your submitted `mp` package directly

The export cell below creates `submission.zip` in the correct layout directly from the notebook runtime.


In [ ]:
from pathlib import Path
import zipfile

required_artifacts = [
    Path("mp/artifacts/checkpoint.pt"),
    Path("mp/artifacts/tokenizer.json"),
]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    print("Warning: missing required artifact files:")
    for path in missing_artifacts:
        print(" -", path)
    print("You can still create the zip now, but Gradescope will expect these files in the final submission.")

submission_zip = Path("submission.zip")
with zipfile.ZipFile(submission_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(Path("mp").rglob("*")):
        if not path.is_file():
            continue
        if "__pycache__" in path.parts or path.suffix == ".pyc" or path.name == ".DS_Store":
            continue
        zf.write(path, arcname=str(path))

print("Wrote", submission_zip.resolve())

try:
    from google.colab import files
except ImportError:
    files = None

if files is not None:
    files.download(str(submission_zip))
else:
    print("Manual download path:", submission_zip.resolve())


## 🔬 Debugging Notes

### Notebook workflow issues

If a test still looks like it is running old code, rerun the relevant `%%writefile` cell and then rerun the test.

If you see `ModuleNotFoundError: No module named mp`, rerun the setup cell and then rerun the download cell.

### Tokenizer tests fail

Check:

- byte-level processing instead of character-level processing
- deterministic tie-breaking in `train`
- merge priority in `encode`
- UTF-8 reconstruction in `decode`

### Causal masking tests fail

Check:

- the mask is applied before normalization
- future positions are blocked instead of earlier positions
- attention score shape is correct before masking

### Cross-entropy tests fail

Check:

- the last dimension is treated as the vocabulary dimension
- the correct-class logit is selected at each position
- the loss is averaged over all batch and sequence positions

### Loss does not decrease

Check:

- `y` is exactly the next-token shift of `x`
- gradients are cleared before backward
- the optimizer step happens
- future-token leakage is not happening in attention


## 📒 Provided Training and Inference Files (Optional)

We provide the training and inference scripts in `mp/run_train.py` and `mp/serve.py`. You do not need to edit these files, but you can read through them to understand how the training loop and generation logic work.

In [ ]:
%%writefile mp/run_train.py
from __future__ import annotations

import itertools
import json
import sys
import threading
import time
from dataclasses import dataclass
from pathlib import Path

import torch

from .model import TransformerConfig, TransformerLM
from .train import evaluate, get_batch, save_checkpoint, train_step

OUT_DIR = None
VOCAB_SIZE = 1024
BATCH_SIZE = 32
SEQ_LEN = 128
MAX_STEPS = 1200
EVAL_INTERVAL = 200
EVAL_ITERS = 20
LEARNING_RATE = 3e-4
N_LAYERS = 4
N_HEADS = 8
D_MODEL = 384
D_FF = 1536
DROPOUT = 0.1
SEED = 0


@dataclass
class _TokenizerState:
    merges: list[tuple[int, int]]


class _FastBPETokenizer:
    """Provided efficient byte-level BPE tokenizer used only by this training script."""

    def __init__(self, merges: list[tuple[int, int]]) -> None:
        self.merges = merges
        self.base_vocab_size = 256
        self.vocab_size = self.base_vocab_size + len(merges)

    @classmethod
    def train(
        cls,
        text: str,
        vocab_size: int,
        min_pair_freq: int = 2,
        progress_callback=None,
    ) -> "_FastBPETokenizer":
        if vocab_size < 256:
            raise ValueError("vocab_size must be >= 256 for byte-level BPE.")
        if min_pair_freq < 1:
            raise ValueError("min_pair_freq must be >= 1.")

        token_ids = list(text.encode("utf-8"))
        merges: list[tuple[int, int]] = []
        next_id = 256
        total_possible_merges = max(0, vocab_size - 256)

        while next_id < vocab_size and len(token_ids) >= 2:
            pair_counts: dict[tuple[int, int], int] = {}
            for i in range(len(token_ids) - 1):
                pair = (token_ids[i], token_ids[i + 1])
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

            if not pair_counts:
                break

            best_pair, best_count = min(
                pair_counts.items(), key=lambda item: (-item[1], item[0])
            )
            if best_count < min_pair_freq:
                break

            merges.append(best_pair)
            token_ids = cls._replace_pair(token_ids, best_pair, next_id)
            if progress_callback is not None:
                progress_callback(len(merges), total_possible_merges, best_pair, best_count)
            next_id += 1

        return cls(merges)

    def encode(self, text: str) -> list[int]:
        token_ids = list(text.encode("utf-8"))
        if not token_ids:
            return []

        for merge_rank, pair in enumerate(self.merges):
            merged_id = self.base_vocab_size + merge_rank
            token_ids = self._replace_pair(token_ids, pair, merged_id)
        return token_ids

    def save(self, path: str) -> None:
        output_path = Path(path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        state = _TokenizerState(merges=self.merges)
        with output_path.open("w", encoding="utf-8") as f:
            json.dump({"merges": state.merges}, f)

    @staticmethod
    def _replace_pair(
        token_ids: list[int],
        pair: tuple[int, int],
        new_token_id: int,
    ) -> list[int]:
        result: list[int] = []
        i = 0
        while i < len(token_ids):
            if (
                i + 1 < len(token_ids)
                and token_ids[i] == pair[0]
                and token_ids[i + 1] == pair[1]
            ):
                result.append(new_token_id)
                i += 2
            else:
                result.append(token_ids[i])
                i += 1
        return result


def _default_corpus_path() -> Path:
    return Path(__file__).resolve().parents[1] / "data" / "tinystories_subset.txt"


def _default_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def _format_duration(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours > 0:
        return f"{hours:d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def _render_tokenizer_progress(
    merges_done: int,
    total_merges: int,
    best_pair: tuple[int, int],
    pair_count: int,
    start_time: float,
) -> str:
    width = 28
    filled = 0 if total_merges == 0 else int(width * merges_done / total_merges)
    bar = "#" * filled + "-" * (width - filled)
    elapsed = time.time() - start_time
    eta_seconds = None
    if merges_done > 0 and total_merges > 0:
        eta_seconds = (elapsed / merges_done) * max(0, total_merges - merges_done)
    eta_text = _format_duration(eta_seconds) if eta_seconds is not None else "--:--"
    return (
        f"[{bar}] merges {merges_done:4d}/{total_merges} "
        f"best_pair={best_pair} freq={pair_count} eta={eta_text}"
    )


def _render_progress(step: int, total_steps: int, loss: float, eta_seconds: float | None) -> str:
    width = 28
    filled = int(width * step / total_steps)
    bar = "#" * filled + "-" * (width - filled)
    eta_text = _format_duration(eta_seconds) if eta_seconds is not None else "--:--"
    return f"[{bar}] step {step:4d}/{total_steps} train_loss={loss:.4f} eta={eta_text}"


def _encode_with_indicator(tokenizer: _FastBPETokenizer, text: str) -> list[int]:
    if not sys.stdout.isatty():
        print("Encoding corpus... this can take a while on large corpora.")
        start_time = time.time()
        token_ids = tokenizer.encode(text)
        print(f"Finished encoding corpus in {_format_duration(time.time() - start_time)}.")
        return token_ids

    done = threading.Event()
    start_time = time.time()

    def render() -> None:
        frames = itertools.cycle(["[#---------------------------]", "[####------------------------]", "[########--------------------]", "[############----------------]", "[################------------]", "[####################--------]", "[########################----]", "[############################]"])
        while not done.wait(0.2):
            elapsed = _format_duration(time.time() - start_time)
            print(
                f"{next(frames)} encoding corpus elapsed={elapsed}",
                end="\r",
                flush=True,
            )

    thread = threading.Thread(target=render, daemon=True)
    thread.start()
    try:
        return tokenizer.encode(text)
    finally:
        done.set()
        thread.join()
        print(" " * 80, end="\r", flush=True)
        print(f"Finished encoding corpus in {_format_duration(time.time() - start_time)}.")


def main() -> None:
    torch.manual_seed(SEED)
    device = _default_device()
    corpus_path = _default_corpus_path()
    use_live_bar = sys.stdout.isatty()

    print(f"Loading corpus from: {corpus_path}", flush=True)
    corpus_text = corpus_path.read_text(encoding="utf-8")
    print("Training tokenizer...", flush=True)
    tokenizer_start = time.time()

    def tokenizer_progress(
        merges_done: int,
        total_merges: int,
        best_pair: tuple[int, int],
        pair_count: int,
    ) -> None:
        line = _render_tokenizer_progress(
            merges_done,
            total_merges,
            best_pair,
            pair_count,
            tokenizer_start,
        )
        if use_live_bar:
            print(line, end="\r", flush=True)
        elif merges_done == 1 or merges_done % 50 == 0 or merges_done == total_merges:
            print(line, flush=True)

    tokenizer = _FastBPETokenizer.train(
        corpus_text,
        vocab_size=VOCAB_SIZE,
        min_pair_freq=2,
        progress_callback=tokenizer_progress,
    )
    if use_live_bar:
        print()
    print("Encoding corpus with learned tokenizer...", flush=True)
    token_ids = _encode_with_indicator(tokenizer, corpus_text)
    all_tokens = torch.tensor(token_ids, dtype=torch.long)

    split_idx = int(0.9 * len(all_tokens))
    train_tokens = all_tokens[:split_idx]
    val_tokens = all_tokens[split_idx:]

    config = TransformerConfig(
        vocab_size=tokenizer.vocab_size,
        max_seq_len=SEQ_LEN,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        d_model=D_MODEL,
        d_ff=D_FF,
        dropout=DROPOUT,
    )

    model = TransformerLM(config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print(f"Starting training on device: {device}", flush=True)
    train_start = time.time()
    for step in range(1, MAX_STEPS + 1):
        x, y = get_batch(train_tokens, BATCH_SIZE, SEQ_LEN, device)
        loss = train_step(model, optimizer, x, y)
        elapsed = time.time() - train_start
        eta_seconds = (elapsed / step) * (MAX_STEPS - step)
        if use_live_bar:
            print(_render_progress(step, MAX_STEPS, loss, eta_seconds), end="\r", flush=True)
        elif step == 1 or step % 50 == 0 or step == MAX_STEPS:
            print(_render_progress(step, MAX_STEPS, loss, eta_seconds), flush=True)

        if step % EVAL_INTERVAL == 0 or step == 1 or step == MAX_STEPS:
            val_loss = evaluate(
                model,
                val_tokens,
                batch_size=BATCH_SIZE,
                seq_len=SEQ_LEN,
                eval_iters=EVAL_ITERS,
            )
            if use_live_bar:
                print()
            print(f"step={step:4d} train_loss={loss:.4f} val_loss={val_loss:.4f}", flush=True)

    if use_live_bar:
        print()

    out_dir = Path(OUT_DIR) if OUT_DIR else Path(__file__).resolve().parent / "artifacts"
    out_dir.mkdir(parents=True, exist_ok=True)

    tokenizer_path = out_dir / "tokenizer.json"
    checkpoint_path = out_dir / "checkpoint.pt"

    tokenizer.save(str(tokenizer_path))
    save_checkpoint(
        path=str(checkpoint_path),
        model=model,
        optimizer=optimizer,
        step=MAX_STEPS,
        config=config,
    )

    print(f"Saved tokenizer to: {tokenizer_path}", flush=True)
    print(f"Saved checkpoint to: {checkpoint_path}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile mp/serve.py
from __future__ import annotations

import argparse

import torch

from .model import TransformerConfig, TransformerLM
from .tokenizer import BPETokenizer


def _build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Generate text from a trained TransformerLM.")
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--tokenizer", type=str, required=True)
    parser.add_argument("--prompt", type=str, required=True)
    parser.add_argument("--max-new-tokens", type=int, default=80)
    parser.add_argument("--temperature", type=float, default=0.7)
    parser.add_argument("--top-k", type=int, default=20)
    parser.add_argument("--device", type=str, default="cpu")
    parser.add_argument("--seed", type=int, default=0)
    return parser


def load_model_and_tokenizer(
    checkpoint_path: str,
    tokenizer_path: str,
    device: str = "cpu",
) -> tuple[TransformerLM, BPETokenizer]:
    """Load model checkpoint and tokenizer."""
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    config = TransformerConfig(**ckpt["config"])
    model = TransformerLM(config)
    model.load_state_dict(ckpt["model_state"])
    model.to(device)
    model.eval()

    tokenizer = BPETokenizer.load(tokenizer_path)
    return model, tokenizer


def main() -> None:
    args = _build_parser().parse_args()
    torch.manual_seed(args.seed)

    model, tokenizer = load_model_and_tokenizer(
        checkpoint_path=args.checkpoint,
        tokenizer_path=args.tokenizer,
        device=args.device,
    )

    prompt_ids = tokenizer.encode(args.prompt)
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=args.device)

    output_ids = model.generate(
        input_ids,
        max_new_tokens=args.max_new_tokens,
        temperature=args.temperature,
        top_k=args.top_k,
    )
    text = tokenizer.decode(output_ids[0].tolist())
    print(text)


if __name__ == "__main__":
    main()
